In [1]:
# For creating noisy image data
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader
import random


class GaussianNoise:
    def __init__(self, mean=0.0, std=0.1):
        self.mean = mean
        self.std = std

    def __call__(self, x):
        noise = torch.randn_like(x) * self.std + self.mean
        return torch.clamp(x + noise, 0.0, 1.0)


class SaltAndPepperNoise:
    def __init__(self, prob=0.05):
        self.prob = prob

    def __call__(self, x):
        noisy = x.clone()
        rand = torch.rand_like(x[0])

        salt = rand < self.prob / 2
        pepper = rand > 1 - self.prob / 2

        noisy[:, salt] = 1.0
        noisy[:, pepper] = 0.0

        return noisy


class GaussianBlur:
    def __init__(self, kernel_size=3, sigma=1.0):
        self.kernel_size = kernel_size
        self.sigma = sigma

    def __call__(self, x):
        return TF.gaussian_blur(
            x,
            kernel_size=[self.kernel_size, self.kernel_size],
            sigma=[self.sigma, self.sigma],
        )


class MixedDistortion:
    def __init__(self):
        self.transforms = [
            GaussianNoise(std=0.1),
            SaltAndPepperNoise(prob=0.05),
            GaussianBlur(kernel_size=3, sigma=1.0),
        ]

    def __call__(self, x):
        distortion = random.choice(self.transforms)
        return distortion(x)


def get_train_transform(distortion_type):
    transform_list = [
        transforms.ToTensor(),
    ]

    if distortion_type == "clean":
        pass
    elif distortion_type == "blur":
        transform_list.append(GaussianBlur(kernel_size=3, sigma=1.0))
    elif distortion_type == "gaussian":
        transform_list.append(GaussianNoise(mean=0.0, std=0.1))
    elif distortion_type == "salt_pepper":
        transform_list.append(SaltAndPepperNoise(prob=0.05))
    elif distortion_type == "mixed":
        transform_list.append(MixedDistortion())
    else:
        raise ValueError(
            "distortion_type must be one of: "
            "'clean', 'blur', 'gaussian', 'salt_pepper', 'mixed'"
        )

    return transforms.Compose(transform_list)



In [2]:
class BasicCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 32x32 -> 16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 16x16 -> 8x8

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 8x8 -> 4x4
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [3]:
def evaluate(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            total_loss += loss.item() * images.size(0)

            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy


def train_cnn_on_distortion(
    distortion_type,
    root="./data",
    batch_size=128,
    num_epochs=10,
    lr=1e-3,
    num_workers=2,
    device=None,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    train_transform = get_train_transform(distortion_type)
    test_transform = transforms.ToTensor()

    train_dataset = datasets.CIFAR10(
        root=root,
        train=True,
        download=True,
        transform=train_transform,
    )

    test_dataset = datasets.CIFAR10(
        root=root,
        train=False,
        download=True,
        transform=test_transform,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=(device == "cuda"),
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(device == "cuda"),
    )

    model = BasicCNN(num_classes=10).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": [],
    }

    for epoch in range(num_epochs):
        model.train()

        total_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * images.size(0)

            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / total
        train_acc = correct / total

        test_loss, test_acc = evaluate(model, test_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)

        print(
            f"Epoch [{epoch + 1}/{num_epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.4f} "
            f"test_loss={test_loss:.4f} "
            f"test_acc={test_acc:.4f}"
        )

    return model, history

def save_model(model: nn.Module, distortion_type: str) -> None:
  torch.save(model.state_dict(), f"model_{distortion_type}.pth")

def load_model(path: str) -> nn.Module:
  model = BasicCNN()
  model.load_state_dict(torch.load(path))
  model.eval()

  return model

In [ ]:
from enum import StrEnum
import json

class DistortionType(StrEnum):
  CLEAN = 'clean'
  BLUR = 'blur'
  GAUSSIAN = 'gaussian'
  SALT_PEPPER = 'salt_pepper'
  MIXED = 'mixed'

# put the type here
d_type =

model, history = train_cnn_on_distortion(d_type)
save_model(model, d_type)

with open(f"model_{d_type}_train_history.json", "w") as f:
  json.dump(history, f)


In [27]:
import os
from torch.utils.data import Subset

MODEL_DIR = "finalProjModelData"
OUT_DIR = "finalProjEmpiricalData"


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 128
DATA_ROOT = "./data"

CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

DISTORTIONS = ["clean", "gaussian", "salt_pepper", "blur", "mixed"]

test_loaders = {}

for distortion in DISTORTIONS:
    test_dataset = datasets.CIFAR10(
        root=DATA_ROOT,
        train=False,
        download=True,
        transform=get_train_transform(distortion),
    )


    test_loaders[distortion] = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=(DEVICE == "cuda"),
    )


In [28]:
MODEL_NAMES = ["clean", "gaussian", "salt_pepper", "blur", "mixed"]

model_paths = {
    "clean": os.path.join(MODEL_DIR, "model_clean.pth"),
    "gaussian": os.path.join(MODEL_DIR, "model_gaussian.pth"),
    "salt_pepper": os.path.join(MODEL_DIR, "model_salt_pepper.pth"),
    "blur": os.path.join(MODEL_DIR, "model_blur.pth"),
    "mixed": os.path.join(MODEL_DIR, "model_mixed.pth"),
}


def load_model(path):
    model = BasicCNN(num_classes=10).to(DEVICE)

    state = torch.load(path, map_location=DEVICE)

    # Handles either a raw state_dict or a checkpoint dict.
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]

    model.load_state_dict(state)
    model.eval()
    return model


models = {
    name: load_model(path)
    for name, path in model_paths.items()
}

print("Loaded models:", list(models.keys()))

Loaded models: ['clean', 'gaussian', 'salt_pepper', 'blur', 'mixed']


In [29]:
from tqdm import tqdm

@torch.no_grad()
def evaluate_model(model, dataloader):
    model.eval()

    all_preds = []
    all_labels = []
    all_probs = []

    total = 0
    correct = 0

    for images, labels in tqdm(dataloader, desc="Evaluating"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())
        all_probs.append(probs.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    all_probs = torch.cat(all_probs).numpy()

    accuracy = correct / total

    return {
        "accuracy": accuracy,
        "preds": all_preds,
        "labels": all_labels,
        "probs": all_probs,
    }

In [30]:
import pandas as pd

results = {}

for model_name, model in models.items():
    print(
          f"Evaluating model trained on '{model_name}'..."
          )
    results[model_name] = {}

    for distortion_name, loader in test_loaders.items():
        print(
                f"  Testing on '{distortion_name}' distortion..."
        )
        eval_result = evaluate_model(model, loader)
        results[model_name][distortion_name] = eval_result

accuracy_matrix = pd.DataFrame(
    {
        distortion_name: {
            model_name: results[model_name][distortion_name]["accuracy"]
            for model_name in MODEL_NAMES
        }
        for distortion_name in DISTORTIONS
    }
)

accuracy_matrix.index.name = "trained_on"
accuracy_matrix.columns.name = "tested_on"

accuracy_matrix

Evaluating model trained on 'clean'...
  Testing on 'clean' distortion...


Evaluating: 100%|██████████| 79/79 [00:01<00:00, 39.82it/s]


  Testing on 'gaussian' distortion...


Evaluating: 100%|██████████| 79/79 [00:02<00:00, 36.67it/s]


  Testing on 'salt_pepper' distortion...


...

  Testing on 'blur' distortion...


Evaluating: 100%|██████████| 79/79 [00:02<00:00, 26.43it/s]


  Testing on 'mixed' distortion...


Evaluating: 100%|██████████| 79/79 [00:02<00:00, 32.95it/s]


tested_on,clean,gaussian,salt_pepper,blur,mixed
trained_on,,,,,
clean,0.7312,0.5641,0.4134,0.6385,0.5313
gaussian,0.6761,0.6878,0.5332,0.5335,0.5867
salt_pepper,0.6998,0.6502,0.6828,0.6403,0.6601
blur,0.6769,0.5949,0.4409,0.6867,0.5778
mixed,0.6912,0.6768,0.6758,0.6749,0.6748


In [32]:
accuracy_matrix_path_csv = os.path.join(OUT_DIR, "empirical_accuracy_matrix.csv")
accuracy_matrix_path_md = os.path.join(OUT_DIR, "empirical_accuracy_matrix.md")

accuracy_matrix.to_csv(accuracy_matrix_path_csv)

with open(accuracy_matrix_path_md, "w") as f:
    f.write("# Empirical Accuracy Matrix\n\n")
    f.write("Rows indicate the distortion used during training. Columns indicate the distortion used during testing.\n\n")
    f.write(accuracy_matrix.to_markdown())

print("Saved:")
print(accuracy_matrix_path_csv)
print(accuracy_matrix_path_md)

Saved:
finalProjEmpiricalData/empirical_accuracy_matrix.csv
finalProjEmpiricalData/empirical_accuracy_matrix.md


In [35]:
from sklearn.metrics import confusion_matrix, precision_score

MATCHED_DISTORTION_MODELS = ["clean", "gaussian", "salt_pepper", "blur", "mixed"]

precision_rows = []

for model_name in MATCHED_DISTORTION_MODELS:
    eval_result = results[model_name][model_name]

    labels = eval_result["labels"]
    preds = eval_result["preds"]

    precisions = precision_score(
        labels,
        preds,
        labels=list(range(10)),
        average=None,
        zero_division=0,
    )

    for class_idx, precision in enumerate(precisions):
        precision_rows.append({
            "model_trained_on": model_name,
            "tested_on": model_name,
            "class_index": class_idx,
            "class_name": CIFAR10_CLASSES[class_idx],
            "precision_P_true_class_given_model_guessed_class": precision,
        })

precision_df = pd.DataFrame(precision_rows)

precision_df

,model_trained_on,tested_on,class_index,class_name,precision_P_true_class_given_model_guessed_class
0,clean,clean,0,airplane,0.809935
1,clean,clean,1,automobile,0.835616
2,clean,clean,2,bird,0.622815
3,clean,clean,3,cat,0.582278
4,clean,clean,4,deer,0.769634
5,clean,clean,5,dog,0.580508
6,clean,clean,6,frog,0.717300
7,clean,clean,7,horse,0.819491
8,clean,clean,8,ship,0.820000
9,clean,clean,9,truck,0.789941


In [36]:
precision_csv_path = os.path.join(OUT_DIR, "matched_distortion_class_precision.csv")
precision_md_path = os.path.join(OUT_DIR, "matched_distortion_class_precision.md")

precision_df.to_csv(precision_csv_path, index=False)

with open(precision_md_path, "w") as f:
    f.write("# Matched-Distortion Class Precision\n\n")
    f.write(
        "Each row estimates "
        "$P(y = g \\mid \\hat{y} = g)$, "
        "the probability that an image is truly class $g$ given that the model guessed class $g$.\n\n"
    )
    f.write(precision_df.to_markdown(index=False))

print("Saved:")
print(precision_csv_path)
print(precision_md_path)

Saved:
finalProjEmpiricalData/matched_distortion_class_precision.csv
finalProjEmpiricalData/matched_distortion_class_precision.md


In [37]:
@torch.no_grad()
def evaluate_softmax_ensemble(models_to_average, dataloader):
    for model in models_to_average:
        model.eval()

    all_preds = []
    all_labels = []
    all_probs = []

    total = 0
    correct = 0

    for images, labels in dataloader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        probs_sum = None

        for model in models_to_average:
            logits = model(images)
            probs = torch.softmax(logits, dim=1)

            if probs_sum is None:
                probs_sum = probs
            else:
                probs_sum += probs

        avg_probs = probs_sum / len(models_to_average)
        preds = torch.argmax(avg_probs, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())
        all_probs.append(avg_probs.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    all_probs = torch.cat(all_probs).numpy()

    accuracy = correct / total

    return {
        "accuracy": accuracy,
        "preds": all_preds,
        "labels": all_labels,
        "probs": all_probs,
    }

In [38]:
specialized_models = [
    models["gaussian"],
    models["salt_pepper"],
    models["blur"],
]

ensemble_vs_mixed_rows = []

for distortion_name, loader in test_loaders.items():
    ensemble_result = evaluate_softmax_ensemble(specialized_models, loader)
    mixed_result = results["mixed"][distortion_name]

    ensemble_acc = ensemble_result["accuracy"]
    mixed_acc = mixed_result["accuracy"]

    ensemble_vs_mixed_rows.append({
        "tested_on": distortion_name,
        "ensemble_accuracy": ensemble_acc,
        "mixed_model_accuracy": mixed_acc,
        "ensemble_minus_mixed": ensemble_acc - mixed_acc,
        "winner": "ensemble" if ensemble_acc > mixed_acc else "mixed" if mixed_acc > ensemble_acc else "tie",
    })

ensemble_vs_mixed_df = pd.DataFrame(ensemble_vs_mixed_rows)

ensemble_vs_mixed_df

,tested_on,ensemble_accuracy,mixed_model_accuracy,ensemble_minus_mixed,winner
0,clean,0.7448,0.6912,0.0536,ensemble
1,gaussian,0.7132,0.6768,0.0364,ensemble
2,salt_pepper,0.6132,0.6758,-0.0626,mixed
3,blur,0.6968,0.6749,0.0219,ensemble
4,mixed,0.6783,0.6748,0.0035,ensemble


In [39]:
ensemble_csv_path = os.path.join(OUT_DIR, "ensemble_vs_mixed_accuracy.csv")
ensemble_md_path = os.path.join(OUT_DIR, "ensemble_vs_mixed_accuracy.md")

ensemble_vs_mixed_df.to_csv(ensemble_csv_path, index=False)

with open(ensemble_md_path, "w") as f:
    f.write("# Ensemble vs Mixed-Distortion Model Accuracy\n\n")
    f.write(
        "The ensemble averages softmax outputs from the Gaussian-trained, "
        "salt-and-pepper-trained, and blur-trained models. "
        "The mixed model is the single model trained on mixed distortions.\n\n"
    )
    f.write(ensemble_vs_mixed_df.to_markdown(index=False))

print("Saved:")
print(ensemble_csv_path)
print(ensemble_md_path)

Saved:
finalProjEmpiricalData/ensemble_vs_mixed_accuracy.csv
finalProjEmpiricalData/ensemble_vs_mixed_accuracy.md


In [41]:
import json

summary = {
    "accuracy_matrix": accuracy_matrix.reset_index().to_dict(orient="records"),
    "matched_distortion_class_precision": precision_df.to_dict(orient="records"),
    "ensemble_vs_mixed": ensemble_vs_mixed_df.to_dict(orient="records"),
}

summary_json_path = os.path.join(OUT_DIR, "empirical_analysis_summary.json")

with open(summary_json_path, "w") as f:
    json.dump(summary, f, indent=2)

print("Saved:")
print(summary_json_path)

Saved:
finalProjEmpiricalData/empirical_analysis_summary.json
